# مشروع HAM10000 و ASISM v2 — شرح الـ pipeline وخريطة الكود

**الحالة يوم 2026-10-04:** مفيش أي تشغيل GPU متوافق عليه لـ v2. القياس (1,000 تدريب) و Stage 4 لسه ما اتعملوش، فالنوتبوك ده بيشرح **التصميم والكود**، مش نتايج v2.

**النوتبوك ده إيه:** مرجع تفتحيه فتفهمي المشروع كله وتشرحي الـ pipeline. لكل مرحلة: شرح بسيط، الملف والدالة المسؤولين عنها فعلًا، أهم الداخل والخارج، ومقطع قصير من الكود لو محتاج.

**النوتبوك ده مش إيه:**

- مش نسخة من الكود. الكود الأصلي في الـ repo هو المرجع، وكل مقطع هنا مقصوص منه وقت بناء النوتبوك.
- مش لتشغيل الـ GPU. أوامر الـ pod في [`ham10000_asism_v2_learned_pod_walkthrough.ipynb`](ham10000_asism_v2_learned_pod_walkthrough.ipynb) و [`docs/ham10000_asism_v2_learned_pod_commands.md`](../docs/ham10000_asism_v2_learned_pod_commands.md).
- خلايا الكود الثلاثة اللي فيه بتشتغل على CPU في ثواني ومفيهاش تدريب.

**مصادر الأرقام:** [`docs/ham10000_v2_experimental_contract.md`](../docs/ham10000_v2_experimental_contract.md) (العقد)، [`docs/ham10000_results_and_limitations.md`](../docs/ham10000_results_and_limitations.md)، [`docs/ham10000_status_2026-10-02.md`](../docs/ham10000_status_2026-10-02.md)، و [`configs/ham10000_asism_v2_ranker.yaml`](../configs/ham10000_asism_v2_ranker.yaml).

النوتبوك بيتبني بـ `python notebooks/build_ham10000_v2_walkthrough.py`. لو الكود اتغير، يتبني تاني؛ ما يتعدّلش بالإيد.

## المحتويات

1. [Dataset / Splits](#s1)
2. [LoRA](#s2)
3. [Synthetic Generation](#s3)
4. [Candidate Pool / Safety](#s4)
5. [Four Signals](#s5)
6. [ASISM V2 — الصورة العامة](#s6)
7. [Ranker Supervision](#s7)
8. [Bootstrap / Lower Bound](#s8)
9. [Adaptive Stopping](#s9)
10. [C / D](#s10)
11. [E4 — تجربة مستقلة لفحص الكمية](#s11)
12. [Stage 4](#s12)
13. [Metrics](#s13)
14. [Final Comparison](#s14)

## Architecture

```
 HAM10000  (10,015 images, 7,470 lesions, 7 classes)
     |
     v
 [1] SPLITS  lesion-level, six parts, frozen as "ham-stratified-v1"
     |
     |-- gen_train (3,586) ----------> [2] LoRA on SDXL  ("ham-lora-v1")
     |                                        |
     |                                        v
     |                                 [3] GENERATION  ->  3,168 synthetic candidates
     |                                        |
     |                                        v
     |                                 [4] CANDIDATE POOL + SAFETY FILTER  (removed 0)
     |                                        |
     |                                        v
     |                                 [5] FOUR SIGNALS per image
     |                                     similarity | IQA | uncertainty | explainability
     |                                        |
     |                                        v
     |          +-------------------- [6] ASISM V2 -----------------------+
     |          |                                                         |
     |-- classifier_train (1,641) --> [7] RANKER SUPERVISION              |
     |-- asism_tuning_heldout ------>     200 designed subsets x 5 seeds  |
     |      (1,377)                       = 1,000 small trainings         |
     |          |                         gate G1 -> fit -> acceptance    |
     |          |                                |                        |
     |          |                         [8] BOOTSTRAP (200 rankers)     |
     |          |                             lower 95% bound per image   |
     |          |                                |                        |
     |          |                         [9] ADAPTIVE STOPPING           |
     |          |                             which images AND how many   |
     |          +--------------------------------|------------------------+
     |                                           v
     |                                    [10] C = selected     D = random, same count per class
     |                                           |
     |-- classifier_train ---------------------->|
     |                                           v
     |                                    [12] STAGE 4  DenseNet-121, 20 seeds
     |                                         A real | B real+all | C real+selected | D real+random
     |                                           |
     |-- classifier_val (1,401) --------------->  monitoring only
     |                                           v
     |                                    [13] METRICS -> [14] COMPARISON
     |                                         confirmatory: C vs D, balanced accuracy
     |
     +-- final_eval_heldout (1,622) ---- protected; Stage 5 waits for the supervisor's policy


 [11] E4 (separate experiment): quantity curve at the Stage 4 recipe, 60 trainings.
      An independent check on HOW MANY. It does not train the ranker and does not set the count.
```

## كل مرحلة في سطر

| # | المرحلة | بتاخد | بتطلّع |
|---|---|---|---|
| 1 | Splits | HAM10000 الخام | ستّ أقسام مجمّدة |
| 2 | LoRA | `gen_train` | `ham-lora-v1` |
| 3 | Generation | الـ LoRA | 3,168 صورة |
| 4 | Pool / Safety | الصور وإشاراتها | الـ pool الآمن (3,168) |
| 5 | Signals | الصور الصناعية | أربع أرقام لكل صورة |
| 6 | ASISM V2 | الـ pool بإشاراته | C و D |
| 7 | Supervision | 200 مجموعة × 5 seeds | درجة منفعة لكل مجموعة، G1، accept |
| 8 | Bootstrap | القياسات | 200 ranker وحد أدنى لكل صورة |
| 9 | Stopping | الحدود الدنيا | أنهي صور وكام |
| 10 | C / D | الاختيار | ملفات C و D |
| 11 | E4 (مستقلة) | الـ pool | منحنى الكمية |
| 12 | Stage 4 | A, B, C, D | 80 classifier |
| 13 | Metrics | التوقعات | سبع مقاييس |
| 14 | Comparison | المقاييس | فواصل ثقة و p مصحّحة |

## خلفية في فقرة: ليه v2؟

النسخة الأولى (v1) اشتغلت من أولها لآخرها واختارت 616 صورة من 3,168. على `final_eval_heldout`: A (حقيقي بس) 0.566، B (حقيقي + الكل) 0.645، C (حقيقي + المختار) 0.612 balanced accuracy. الاختبار التأكيدي C ضد B طلع −0.033 (p = 0.047)، يعني الاختيار ما غلبش «استخدم الكل». النتيجة دي مجمّدة وما بتتعدّلش.

التشخيص بعدها طلّع ثلاث مشاكل، وكل واحدة اتحوّلت لقاعدة في v2:

| المشكلة في v1 | القاعدة في v2 |
|---|---|
| C فيها 616 و B فيها 3,168: المقارنة خلطت «أنهي صور» مع «كام صورة» | حالة **D**: عشوائي بنفس عدد C لكل class |
| درجة المنفعة اللي اتعلم منها الـ ranker كانت أغلبها ضوضاء (تشغيلة واحدة لكل مجموعة) | **5 تكرارات** لكل مجموعة، وبوابة **G1** قبل أي تعلّم |
| العدد اتحدد عمليًا بحد أدنى يدوي | **قاعدة وقوف**: العدد ناتج من النموذج، مفيش K ولا نسبة |

كود v1 وتجارب التشخيص لسه في الـ repo، بس مش جزء من المسار اللي النوتبوك ده بيشرحه.

## فحص خريطة الكود

الخلية دي بتتأكد إن كل ملف ودالة النوتبوك بيذكرهم موجودين في الـ checkout المفتوح. قراءة بس، من غير torch ولا GPU.

In [ ]:
import ast
from pathlib import Path

CODE_MAP = {
    "scripts/data/ham10000/01_build_splits.py": [
        "partition_groups_stratified",
        "audit_splits"
    ],
    "scripts/data/ham10000/02_preprocess_images.py": [
        "letterbox_resize_rgb"
    ],
    "scripts/utils/splits.py": [
        "assert_final_eval_access_allowed"
    ],
    "configs/splits_ham10000.yaml": [],
    "scripts/data/ham10000/04_prepare_lora_inputs.py": [
        "prepare_split"
    ],
    "scripts/train/ham10000_train_lora_sdxl.py": [
        "train_loop",
        "training_step",
        "export_final"
    ],
    "configs/ham10000_stage1.yaml": [],
    "scripts/generate/ham10000_recipes.py": [
        "rarity_quotas",
        "derive_image_seed"
    ],
    "scripts/generate/ham10000_generate_synthetic_images.py": [
        "run_generation",
        "validate_generation_outputs"
    ],
    "scripts/generate/ham10000_standardize_generated.py": [
        "standardize_directory"
    ],
    "configs/ham10000_stage2.yaml": [],
    "scripts/asism/candidate_pool.py": [
        "unsafe_candidates"
    ],
    "scripts/asism_v2/supervision.py": [
        "load_signal_table",
        "assign_roles",
        "build_plan",
        "check_plan"
    ],
    "scripts/asism_v2/features.py": [
        "validate_frame",
        "TrainingStandardizer"
    ],
    "scripts/asism/ham10000_01_compute_signals.py": [
        "run_similarity",
        "run_iqa",
        "run_uncertainty_and_agreement",
        "run_explainability"
    ],
    "scripts/asism/ham10000_02_gonogo.py": [
        "check_redundancy",
        "decide"
    ],
    "configs/ham10000_asism_v2_ranker.yaml": [],
    "scripts/asism_v2/prereg.py": [
        "load_prereg"
    ],
    "scripts/followup/ham10000_asism_v2_utility.py": [
        "run_plan",
        "run_measure",
        "run_gate"
    ],
    "scripts/followup/ham10000_asism_v2_learned_select.py": [
        "run_accept",
        "run_fit",
        "run_select",
        "run_stability"
    ],
    "scripts/asism_v2/contracts.py": [
        "write_new_json",
        "validate_measurements"
    ],
    "scripts/asism/ham10000_03_build_utility_subsets.py": [
        "_measure"
    ],
    "scripts/asism_v2/gates.py": [
        "reliability_gate",
        "stratified_spearman",
        "acceptance",
        "selection_stability"
    ],
    "scripts/asism_v2/pipeline.py": [
        "AdditiveUtilityRanker",
        "fit_ranker",
        "_train"
    ],
    "scripts/asism_v2/stopping.py": [
        "progressive_select",
        "_utility"
    ],
    "scripts/asism_v2/persist.py": [
        "save_fitted",
        "load_fitted"
    ],
    "scripts/asism_v2/selection_files.py": [
        "build_selection",
        "draw_matched_random",
        "write_selection"
    ],
    "scripts/utils/ham10000_conditions.py": [
        "ConditionProtocol"
    ],
    "scripts/followup/ham10000_e4_quantity_curve.py": [
        "build_chains",
        "plan_cells",
        "check_stage4_recipe",
        "decide"
    ],
    "scripts/followup/ham10000_e4_consequences.py": [
        "q_star"
    ],
    "docs/ham10000_e4_quantity_design.md": [],
    "scripts/classify/ham10000_train_conditions.py": [
        "build_condition_records",
        "resolve_synthetic_manifest",
        "run"
    ],
    "scripts/utils/ham10000_classifier.py": [
        "train_classifier"
    ],
    "scripts/classify/ham10000_aggregate_conditions.py": [
        "check_fairness",
        "check_selection_outcome"
    ],
    "configs/ham10000_asism_v2_learned_stage4.yaml": [],
    "scripts/utils/ham10000_metrics.py": [
        "balanced_accuracy",
        "macro_average_precision",
        "multiclass_brier_score",
        "top_label_ece"
    ],
    "scripts/eval/ham10000_compare_conditions.py": [
        "metric_functions",
        "compare"
    ],
    "scripts/utils/metrics.py": [
        "patient_level_bootstrap",
        "paired_bootstrap_difference",
        "holm_bonferroni",
        "benjamini_hochberg"
    ],
    "scripts/followup/ham10000_asism_v2_compare.py": [
        "run"
    ],
    "scripts/eval/ham10000_seed_robustness.py": [
        "robustness"
    ]
}


def find_repo_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "scripts").is_dir() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook from the project checkout.")


def defined_names(path):
    names = set()

    def walk(body, prefix=""):
        for node in body:
            if isinstance(node, (ast.FunctionDef, ast.ClassDef)):
                names.add(prefix + node.name)
                if isinstance(node, ast.ClassDef):
                    walk(node.body, prefix + node.name + ".")

    walk(ast.parse(path.read_text(encoding="utf-8")).body)
    return names


REPO = find_repo_root()
missing = []
for relative, wanted in CODE_MAP.items():
    path = REPO / relative
    if not path.is_file():
        missing.append(relative)
    elif wanted:
        missing += [f"{relative}:{name}" for name in wanted if name not in defined_names(path)]
print(f"{len(CODE_MAP)} files, {sum(map(len, CODE_MAP.values()))} functions and classes referenced")
print("all present" if not missing else "MISSING (rebuild the notebook): " + ", ".join(missing))

<a id="s1"></a>

---
## 1. Dataset / Splits

**الفكرة:** HAM10000 فيها 10,015 صورة لـ 7,470 lesion و 7 أنواع (`nv`، `mel`، `bkl`، `bcc`، `akiec`، `vasc`، `df`). نفس الـ lesion ممكن يتصور أكتر من مرة، فالتقسيم بيتعمل على مستوى الـ **lesion**: لو صورتين لنفس الـ lesion وقعوا في قسمين مختلفين، كل الأرقام اللي بعد كده تبقى منفوخة. بعد التقسيم فيه audit بيثبت على الملفات المكتوبة فعلًا إن مفيش تسريب، ولو لقى تسريب السكريبت بيفشل.

| القسم | عدد الصور | الاستخدام المسموح |
|---|---|---|
| `gen_train` | 3,586 | تدريب الـ LoRA، ومرجع للإشارات |
| `gen_val` | 388 | متابعة تدريب الـ LoRA بس |
| `classifier_train` | 1,641 | الجزء الحقيقي في كل classifier |
| `classifier_val` | 1,401 | متابعة Stage 4. **ممنوع في اختيار الصور** |
| `asism_tuning_heldout` | 1,377 | قياس المنفعة بس |
| `final_eval_heldout` | 1,622 | التقييم النهائي بس، ومحمي في الكود |

**ليه ستّ أقسام:** كل قرار بيتاخد على داتا غير اللي بيتحكم عليه بيها.

**المعالجة:** الصور بتتحط على مربع 512×512 بـ letterbox (تصغير + حشو رمادي)، من غير قص عشان الـ lesion اللي على الطرف ما يتقطعش، ومن غير تحويل لأبيض وأسود لأن اللون أهم علامة في الـ dermoscopy.

| بيعمل إيه | فين في الكود |
|---|---|
| تقسيم الـ lesions على الأقسام مع الحفاظ على نسب الأنواع | `partition_groups_stratified` — [`scripts/data/ham10000/01_build_splits.py:159`](../scripts/data/ham10000/01_build_splits.py) |
| إثبات إن مفيش lesion ولا صورة في قسمين | `audit_splits` — [`scripts/data/ham10000/01_build_splits.py:180`](../scripts/data/ham10000/01_build_splits.py) |
| الـ letterbox | `letterbox_resize_rgb` — [`scripts/data/ham10000/02_preprocess_images.py:58`](../scripts/data/ham10000/02_preprocess_images.py) |
| حماية `final_eval_heldout` | `assert_final_eval_access_allowed` — [`scripts/utils/splits.py:63`](../scripts/utils/splits.py) |
| الإعدادات (النسب، الـ seed 42، اسم التقسيم) | [`configs/splits_ham10000.yaml`](../configs/splits_ham10000.yaml) |

**الداخل:** ملف الـ metadata بتاع HAM10000 والصور الخام.
**الخارج:** `data/ham10000/processed/splits/ham-stratified-v1/<split>.csv` و `processed/images/ham-stratified-v1/<split>/`.

**من الكود الفعلي** — [`scripts/data/ham10000/01_build_splits.py`](../scripts/data/ham10000/01_build_splits.py)، السطور 185–195:

```python
overlaps = {"lesion": {}, "image": {}}
names = list(frames)

for column, key in ((group_column, "lesion"), (image_id_column, "image")):
    for i, left in enumerate(names):
        for right in names[i + 1 :]:
            shared = set(frames[left][column]) & set(frames[right][column])
            # ASCII " & ", not U+2229: a Windows cp1252 console cannot encode the set symbol
            # and the audit print crashed on it before writing anything.
            overlaps[key][f"{left} & {right}"] = sorted(shared)[:10]
            overlaps[key][f"{left} & {right} count"] = len(shared)
```

<a id="s2"></a>

---
## 2. LoRA

**الفكرة:** SDXL نموذج بيرسم صور من نص، ومتدرب على صور عامة. تدريبه كله غالي جدًا، فبنستخدم **LoRA**: أوزان SDXL بتفضل مجمّدة، وبنضيف جنبها مصفوفات صغيرة بتتعلم «الفرق» المطلوب بس. الـ `rank` هو حجم المصفوفات دي.

| البند | القيمة (مجمّدة) |
|---|---|
| النموذج الأساسي | SDXL base 1.0 |
| LoRA | rank 32، alpha 32، على الـ UNet بس |
| التدريب | 8,000 خطوة، 768×576، learning rate 1e-4 (cosine) |
| الداتا | `gen_train` للتدريب، `gen_val` للمتابعة |
| الناتج | `ham-lora-v1/final` |

**تفصيلة مهمة:** الـ LoRA بيتدرب على الصور **من غير الحشو الرمادي** وبنسبتها الأصلية 4:3. لو اتدرب على المربعات المحشوة هيتعلم يرسم شرايط رمادية.

| بيعمل إيه | فين في الكود |
|---|---|
| تجهيز صور التدريب (من غير حشو) والـ captions | `prepare_split` — [`scripts/data/ham10000/04_prepare_lora_inputs.py:72`](../scripts/data/ham10000/04_prepare_lora_inputs.py) |
| حلقة التدريب | `train_loop` — [`scripts/train/ham10000_train_lora_sdxl.py:455`](../scripts/train/ham10000_train_lora_sdxl.py) |
| خطوة تدريب واحدة (إضافة ضوضاء وتوقّعها) | `training_step` — [`scripts/train/ham10000_train_lora_sdxl.py:432`](../scripts/train/ham10000_train_lora_sdxl.py) |
| حفظ الـ adapter النهائي | `export_final` — [`scripts/train/ham10000_train_lora_sdxl.py:490`](../scripts/train/ham10000_train_lora_sdxl.py) |
| الإعدادات | [`configs/ham10000_stage1.yaml`](../configs/ham10000_stage1.yaml) |

**الداخل:** صور `gen_train` والـ captions (اسم المرض في نص).
**الخارج:** ملف الـ LoRA adapter، ومعاه provenance (hash الإعدادات والتقسيم).

**حدود معروفة:** generator واحد بـ seed واحد و snapshot واحد، ومفيش إثبات مستقل إن كل صورة فعلًا شكل المرض المطلوب.

<a id="s3"></a>

---
## 3. Synthetic Generation

**الفكرة:** لكل class بنطلب من SDXL + LoRA عدد من الصور. الأنواع النادرة بتاخد أكتر، بس بحساب: المعادلة بتستخدم الجذر التربيعي (`rarity_exponent = 0.5`) عشان ما نطلبش من الـ generator تنوّع ما شافوش في class فيه حوالي 100 صورة حقيقية بس.

`target = 400 × (median_count / class_count) ^ 0.5`، محصور بين 100 و 1,500.

الناتج **3,168 صورة**:

| nv | mel | bkl | bcc | akiec | vasc | df |
|---|---|---|---|---|---|---|
| 112 | 276 | 273 | 400 | 486 | 736 | 885 |

دي أعداد **candidates** مش أعداد مختارة: ASISM هو اللي بيقرر بعد كده. و v2 مش بيولّد صور جديدة.

كل صورة ليها seed خاص بيها محسوب من `sha256(seed:recipe_id)`، فأي صورة ممكن تتعاد لوحدها. الإعدادات: 40 خطوة، guidance 7.0، 768×576. بعد التوليد الصور بتعدّي على نفس دالة الـ letterbox بتاعة الصور الحقيقية.

| بيعمل إيه | فين في الكود |
|---|---|
| حساب عدد الصور لكل class | `rarity_quotas` — [`scripts/generate/ham10000_recipes.py:31`](../scripts/generate/ham10000_recipes.py) |
| seed لكل صورة | `derive_image_seed` — [`scripts/generate/ham10000_recipes.py:118`](../scripts/generate/ham10000_recipes.py) |
| التوليد نفسه | `run_generation` — [`scripts/generate/ham10000_generate_synthetic_images.py:335`](../scripts/generate/ham10000_generate_synthetic_images.py) |
| فحص كل صورة ناتجة ضد الـ manifest | `validate_generation_outputs` — [`scripts/generate/ham10000_generate_synthetic_images.py:425`](../scripts/generate/ham10000_generate_synthetic_images.py) |
| توحيد الهندسة مع الصور الحقيقية | `standardize_directory` — [`scripts/generate/ham10000_standardize_generated.py:29`](../scripts/generate/ham10000_standardize_generated.py) |
| الإعدادات | [`configs/ham10000_stage2.yaml`](../configs/ham10000_stage2.yaml) |

**الداخل:** الـ LoRA adapter، وأعداد الأنواع في `gen_train` بس.
**الخارج:** `outputs/ham10000/stage2/ham-stratified-v1/all_candidates.csv` (ده كمان مدخل الحالة B) والصور.

**حدود معروفة (قياسات، مش أخطاء كود):** تنوّع أقل من الحقيقي؛ 84.8% من `mel` الصناعي الـ classifier المساعد قراه `nv`؛ و `bkl` الصناعي كل الـ judges اللي اتجربوا بيتعرفوا عليه وحش.

**من الكود الفعلي** — [`scripts/generate/ham10000_recipes.py`](../scripts/generate/ham10000_recipes.py)، السطور 51–64:

```python
present = [count for count in counts.values() if count > 0]
if not present:
    raise ValueError("no class has any real images; cannot derive quotas")
median = float(np.median(present))

quotas = {}
for label in CLASSIFIER_TARGET_LABELS:
    count = counts.get(label, 0)
    if count <= 0:
        quotas[label] = int(max_per_class)
        continue
    scaled = base_quota * (median / count) ** rarity_exponent
    quotas[label] = int(np.clip(round(scaled), min_per_class, max_per_class))
return quotas
```

<a id="s4"></a>

---
## 4. Candidate Pool / Safety

**الفكرة:** قبل أي ترتيب، فلتر أمان بيشيل نوعين بس: الصورة اللي ما اتفتحتش أو ما اتقاستش (`iqa_valid` false)، والصورة اللي نسخة شبه مطابقة لصورة حقيقية (تشابه ≥ 0.95). الفلتر «fail closed»: الصورة اللي مالهاش صف في ملف الإشارات بتتعامل كإنها غير آمنة. علامات الجودة التانية (blur، تباين، حواف) **مش** شروط أمان؛ دي بتدخل كإشارة.

**النتيجة الفعلية:** الفلتر شال **صفر** صورة، فالـ 3,168 كلهم آمنين.

**التجميد:** ملف الـ candidates متثبّت بـ sha256 في الكود (`bf8047b5…`). أي نسخة تانية من الملف بتترفض، فما ينفعش مجموعة الصور تتغير من غير ما حد ياخد باله.

| بيعمل إيه | فين في الكود |
|---|---|
| قاعدة الأمان (المشتركة بين كل المراحل) | `unsafe_candidates` — [`scripts/asism/candidate_pool.py:13`](../scripts/asism/candidate_pool.py) |
| تحميل الـ pool لـ v2: فحص الـ hash، دمج الإشارات الأربعة، تطبيق الأمان | `load_signal_table` — [`scripts/asism_v2/supervision.py:48`](../scripts/asism_v2/supervision.py) |
| فحص إن الإشارات كاملة ومفيش قيم ناقصة | `validate_frame` — [`scripts/asism_v2/features.py:15`](../scripts/asism_v2/features.py) |

**الداخل:** `all_candidates.csv` وملفات الإشارات الأربعة (`*_scores.parquet`).
**الخارج:** جدول واحد في الذاكرة: `image_id`، `dx`، والإشارات الأربعة لكل صورة آمنة، وتقرير بالـ hashes.

**من الكود الفعلي** — [`scripts/asism_v2/supervision.py`](../scripts/asism_v2/supervision.py)، السطور 71–78:

```python
removed = {}
if safety.get("reject_invalid_iqa"):
    invalid = artifacts["iqa"].index[~artifacts["iqa"]["iqa_valid"].astype(bool)]
    removed.update({i: "invalid_iqa" for i in invalid})
if safety.get("reject_near_duplicates"):
    near = artifacts["similarity"].index[artifacts["similarity"]["novelty_is_near_duplicate"].astype(bool)]
    removed.update({i: "near_duplicate" for i in near})
safe = pool.drop(index=list(removed)).reset_index(drop=True)
```

<a id="s5"></a>

---
## 5. Four Signals

**الفكرة:** كل صورة صناعية بتاخد أربع أرقام. كل إشارة بتتحفظ في ملف لوحدها ومعاها provenance، عشان أي واحدة تتعاد أو تتراجع من غير ما تبوّظ الباقي.

| الإشارة | العمود المستخدم | بتقيس إيه | إزاي |
|---|---|---|---|
| Similarity | `similarity_knn_mean` | شبه الصور الحقيقية من نفس الـ class | DINOv2 مجمّد، متوسط التشابه مع أقرب 15 صورة من `gen_train` |
| IQA | `iqa_composite` | جودة الصورة | حدّة، تباين، وخمس علامات عيوب |
| Uncertainty | `uncertainty_mutual_information` | الـ classifier محتار قد إيه | MC dropout، 20 مرة |
| Explainability | `explainability_calibrated_typicality` | الـ classifier بيبص فين، وهل ده طبيعي للـ class | Grad-CAM مقارنة بمرجع من `gen_train` |

**إشارة خامسة اتشالت (Agreement):** محتاجة «حَكَم» موثوق يقول الصورة من أنهي class. الحكمين اللي اتجربوا فشلوا في فحوصهم المكتوبة مسبقًا، فما اتستخدمتش.

**Go/No-Go:** فحص قبل أي تعلّم: كل إشارة سليمة تقنيًا، مفيش قيم ناقصة، ومش تكرار لإشارة تانية (الارتباط جوه الـ class أقل من 0.90). الأربعة عدّوا. النجاح هنا معناه إن الإشارات **مختلفة عن بعض**، مش إنها مفيدة؛ ده اللي v2 بيختبره.

**في v2 مفيش اتجاه مفروض:** محدش بيقول «uncertainty الأعلى أحسن» أو العكس. الـ ranker بيتعلم وزن واتجاه كل إشارة لكل class. والتطبيع (طرح المتوسط والقسمة على الانحراف) بيتحسب من صور الـ train بس.

| بيعمل إيه | فين في الكود |
|---|---|
| Similarity | `run_similarity` — [`scripts/asism/ham10000_01_compute_signals.py:333`](../scripts/asism/ham10000_01_compute_signals.py) |
| IQA | `run_iqa` — [`scripts/asism/ham10000_01_compute_signals.py:184`](../scripts/asism/ham10000_01_compute_signals.py) |
| Uncertainty (ومعاها Agreement في نفس الدالة) | `run_uncertainty_and_agreement` — [`scripts/asism/ham10000_01_compute_signals.py:402`](../scripts/asism/ham10000_01_compute_signals.py) |
| Explainability | `run_explainability` — [`scripts/asism/ham10000_01_compute_signals.py:474`](../scripts/asism/ham10000_01_compute_signals.py) |
| فحص التكرار بين الإشارات | `check_redundancy` — [`scripts/asism/ham10000_02_gonogo.py:364`](../scripts/asism/ham10000_02_gonogo.py) |
| قرار كل إشارة | `decide` — [`scripts/asism/ham10000_02_gonogo.py:487`](../scripts/asism/ham10000_02_gonogo.py) |
| التطبيع على صور الـ train بس | `TrainingStandardizer` — [`scripts/asism_v2/features.py:26`](../scripts/asism_v2/features.py) |

**الداخل:** الصور الصناعية، `gen_train` كمرجع، والـ classifier المساعد (V3a).
**الخارج:** `similarity_scores.parquet`، `iqa_scores.parquet`، `uncertainty_scores.parquet`، `explainability_scores.parquet`، ولكل واحد `.provenance.json`.

<a id="s6"></a>

---
## 6. ASISM V2 — الصورة العامة

**الفكرة:** ASISM v2 بيجاوب على سؤالين مع بعض: **أنهي صور** و**كام صورة**، من غير ما حد يحدد K أو نسبة. بيعمل كده على سبع خطوات، كل واحدة بترفض تشتغل لو اللي قبلها ما نجحتش:

| # | الخطوة | CPU / GPU | الملف الناتج |
|---|---|---|---|
| 1 | **plan**: رسم 200 مجموعة مصممة | CPU | `utility_plan.json` |
| 2 | **measure**: 1,000 تدريب صغير | **GPU** (مقفول) | `utility_runs_fit.jsonl`، `utility_runs_test.jsonl` |
| 3 | **gate (G1)**: هل القياس ثابت؟ | CPU | `g1_reliability.json` |
| 4 | **accept**: هل الـ ranker بيتوقع صح على مجموعات ما شافهاش؟ | CPU | `ranker_acceptance.json` |
| 5 | **fit**: الـ ranker و 200 نسخة bootstrap | CPU | `ranker_fit_seed42.json` |
| 6 | **select**: قاعدة الوقوف، وكتابة C و D | CPU | `c_selected.csv`، `d_selected_seed*.csv`، manifest |
| 7 | **stability**: نفس الاختيار بخمس fit seeds | CPU | `selection_stability.json` |

**التسجيل المسبق (pre-registration):** كل رقم بيأثر على النتيجة مكتوب في `configs/ham10000_asism_v2_ranker.yaml` ومنسوخ في الكود. لو الملف اختلف عن النسخة اللي في الكود، التحميل بيرفض. فما ينفعش رقم يتغير بتعديل ملف الإعدادات بس.

**القفل:** خطوة measure بترفض طول ما `MEASUREMENT_APPROVED = False`، حتى لو الـ flag اتكتب في الأمر.

| بيعمل إيه | فين في الكود |
|---|---|
| الإعدادات المعتمدة | [`configs/ham10000_asism_v2_ranker.yaml`](../configs/ham10000_asism_v2_ranker.yaml) |
| رفض أي قيمة مختلفة عن المعتمد | `load_prereg` — [`scripts/asism_v2/prereg.py:64`](../scripts/asism_v2/prereg.py) |
| plan | `run_plan` — [`scripts/followup/ham10000_asism_v2_utility.py:72`](../scripts/followup/ham10000_asism_v2_utility.py) |
| measure | `run_measure` — [`scripts/followup/ham10000_asism_v2_utility.py:112`](../scripts/followup/ham10000_asism_v2_utility.py) |
| gate | `run_gate` — [`scripts/followup/ham10000_asism_v2_utility.py:208`](../scripts/followup/ham10000_asism_v2_utility.py) |
| accept | `run_accept` — [`scripts/followup/ham10000_asism_v2_learned_select.py:79`](../scripts/followup/ham10000_asism_v2_learned_select.py) |
| fit | `run_fit` — [`scripts/followup/ham10000_asism_v2_learned_select.py:88`](../scripts/followup/ham10000_asism_v2_learned_select.py) |
| select | `run_select` — [`scripts/followup/ham10000_asism_v2_learned_select.py:99`](../scripts/followup/ham10000_asism_v2_learned_select.py) |
| stability | `run_stability` — [`scripts/followup/ham10000_asism_v2_learned_select.py:116`](../scripts/followup/ham10000_asism_v2_learned_select.py) |
| الكتابة مرة واحدة (exclusive create) | `write_new_json` — [`scripts/asism_v2/contracts.py:15`](../scripts/asism_v2/contracts.py) |

**الداخل:** الـ pool الآمن بإشاراته، `classifier_train`، `asism_tuning_heldout`.
**الخارج:** كله في `outputs/ham10000/stage3_asism_v2_ranker/ham-stratified-v1/`.

**من الكود الفعلي** — [`scripts/followup/ham10000_asism_v2_utility.py`](../scripts/followup/ham10000_asism_v2_utility.py)، السطور 112–120:

```python
def run_measure(namespace: str, device: str | None, flag_given: bool, candidates: Path | None = None,
                out_dir: Path | None = None, measure_fn: Callable | None = None,
                inputs: dict | None = None) -> dict:
    """Train and score every pending cell. `measure_fn` and `inputs` exist for the CPU tests."""
    if not MEASUREMENT_APPROVED:
        raise UtilityError("the utility measurement is a GPU run that has NOT been approved "
                           "(contract §8: 'the measurement needs its own approval'). Nothing is trained.")
    if not flag_given:
        raise UtilityError(f"this phase trains real models on a GPU; pass {REAL_MODELS_FLAG}")
```

<a id="s7"></a>

---
## 7. Ranker Supervision

**الفكرة:** الـ ranker محتاج «إجابات صح» يتعلم منها. الإجابة هي: **لو ضفت المجموعة دي من الصور الصناعية للتدريب، الـ classifier بيبقى كويس قد إيه؟** بنقيس ده فعلًا بتدريب classifier صغير.

**1) المجموعات مصممة، مش عشوائية.** في v1 المجموعات كانت عشوائية وبحجم واحد، فكانت شبه بعض والفرق بينها أقل من الضوضاء. هنا:

- كل صورة بتاخد **دور** واحد: train (60%) أو validation (20%) أو test (20%)، جوه الـ class بتاعها. المجموعة فيها صور من دور واحد بس، فالثلاث مجموعات ما بيتشاركوش في أي صورة.
- **الأحجام مختلفة:** 125 / 250 / 500 / 1,000 للـ train، و 125 / 250 / 500 للباقي.
- **نسب الـ classes مختلفة** من مجموعة للتانية (Dirichlet).
- **نص المجموعات «مايلة»:** مسحوبة من النص الأعلى أو الأدنى لإشارة واحدة، عشان الإشارات يبقى ليها أثر ممكن يتقاس.
- العدد: 120 train + 40 validation + 40 test = **200 مجموعة**.

**2) القياس.** لكل مجموعة: تدريب DenseNet صغير (224 px، 300 خطوة) على `classifier_train` + المجموعة، وقياس macro AUROC على `asism_tuning_heldout`. بيتكرر بـ **5 seeds** (42–46)، والدرجة هي المتوسط. المجموع 200 × 5 = **1,000 تدريب**. الدرجة مطلقة، من غير طرح خط أساس.

**3) بوابة G1 (قبل أي تعلّم).** بتسأل: الفرق بين المجموعات حقيقي ولا ضوضاء seeds؟ بتحسب ثبات متوسط الـ 5 تكرارات، ولازم يبقى **≥ 0.80**. بتتحسب على مجموعات train و validation بس؛ الـ test لسه ما اتفتحش. ملحوظة مكتوبة في الكود: الرقم ده فيه أثر الحجم، ونفس الرقم جوه كل حجم بيتكتب جنبه بس مش هو اللي بيحكم.

**4) النموذج.**

`U(S) = b + λ · log(1 + Σ wᵢ)` ،  `wᵢ = 1 + score(xᵢ, classᵢ) + class term`

`wᵢ` هو «العدد الفعلي» للصورة: 1 صورة متوسطة، أكتر من 1 صورة بتتحسب بأكتر، صفر ما بتضيفش، وسالب بتضر. الـ `score` مجموع الإشارات الأربعة مضروبة في أوزان بتتعلم **لكل class**.

**5) accept (مرة واحدة).** بعد الـ fit، بنقرا مجموعات الـ test مرة واحدة والملف بيتكتب exclusive، فما ينفعش تتقري وتتعدّل وتتقري تاني. شرطين لازم الاتنين:

- ارتباط Spearman **جوه كل حجم** بين التوقع والقياس ≥ 0.50، و p ≤ 0.05 (10,000 permutation).
- خطأ الـ ranker على الـ test أقل من نموذج «الحجم والـ class بس» (من غير إشارات).

كمان بيتسجل للمقارنة: similarity لوحدها، والـ composite متساوي الأوزان.

**لو G1 أو accept فشل: الترتيب المتعلَّم ما بيتستخدمش، ودي النتيجة اللي بتتكتب.**

| بيعمل إيه | فين في الكود |
|---|---|
| توزيع الأدوار جوه كل class | `assign_roles` — [`scripts/asism_v2/supervision.py:109`](../scripts/asism_v2/supervision.py) |
| رسم الـ 200 مجموعة | `build_plan` — [`scripts/asism_v2/supervision.py:145`](../scripts/asism_v2/supervision.py) |
| فحص إن التصميم يقدر يحدد أثر الحجم والـ class | `check_plan` — [`scripts/asism_v2/supervision.py:202`](../scripts/asism_v2/supervision.py) |
| التدريب الصغير نفسه | `_measure` — [`scripts/asism/ham10000_03_build_utility_subsets.py:242`](../scripts/asism/ham10000_03_build_utility_subsets.py) |
| اشتراط الشبكة الكاملة (كل مجموعة × كل seed) | `validate_measurements` — [`scripts/asism_v2/contracts.py:35`](../scripts/asism_v2/contracts.py) |
| G1 | `reliability_gate` — [`scripts/asism_v2/gates.py:44`](../scripts/asism_v2/gates.py) |
| النموذج | `AdditiveUtilityRanker` — [`scripts/asism_v2/pipeline.py:30`](../scripts/asism_v2/pipeline.py) |
| الـ fit (مع early stopping على الـ validation) | `fit_ranker` — [`scripts/asism_v2/pipeline.py:135`](../scripts/asism_v2/pipeline.py) |
| Spearman جوه كل حجم | `stratified_spearman` — [`scripts/asism_v2/gates.py:79`](../scripts/asism_v2/gates.py) |
| accept | `acceptance` — [`scripts/asism_v2/gates.py:112`](../scripts/asism_v2/gates.py) |

**الداخل:** الـ pool بإشاراته، الخطة، والـ 1,000 قياس.
**الخارج:** `g1_reliability.json`، `ranker_acceptance.json`.

**من الكود الفعلي** — [`scripts/asism_v2/pipeline.py`](../scripts/asism_v2/pipeline.py)، السطور 45–60:

```python
def score(self, x: torch.Tensor, classes: torch.Tensor) -> torch.Tensor:
    return (self.weights[classes] * x).sum(-1)

def image_weight(self, x: torch.Tensor, classes: torch.Tensor) -> torch.Tensor:
    """The image's effective count w_i."""
    return 1 + self.score(x, classes) + (self.class_mix - self.class_mix.mean())[classes]

def forward(self, x: torch.Tensor, classes: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    if x.ndim != 3 or classes.shape != mask.shape or classes.shape != x.shape[:2]:
        raise ValueError("Invalid batched subset dimensions")
    if mask.dtype != torch.bool or not mask.any(dim=1).all():
        raise ValueError("Expected nonempty Boolean subset mask")
    if not torch.isfinite(x[mask]).all():
        raise ValueError("Nonfinite active features")
    total = self.image_weight(x, classes).masked_fill(~mask, 0).sum(1)
    return self.intercept + self.log_count * torch.log1p(torch.relu(total))
```

**من الكود الفعلي** — [`scripts/asism_v2/gates.py`](../scripts/asism_v2/gates.py)، السطور 51–59:

```python
groups = {sid: np.asarray(v, dtype=float) for sid, v in values.items()}
overall = _anova(list(groups.values()))
within_size = {}
for size in sorted(set(sizes[sid] for sid in groups)):
    members = [groups[sid] for sid in groups if sizes[sid] == size]
    within_size[str(size)] = _anova(members) if len(members) >= 2 else None
return {"gate": "G1", "threshold": float(threshold), **overall,
        "passed": bool(overall["reliability_of_mean"] >= threshold),
        "within_size_not_gating": within_size}
```

### مثال توضيحي 1: شكل معادلة المنفعة

**الأرقام هنا مخترعة للشرح، مش قياسات.** الخلية بتحسب `U(S) = b + λ·log(1 + Σw)` لثلاث أنواع من الصور: متوسطة (`w = 1`)، قوية (`w = 1.5`)، وضعيفة (`w = 0.5`).

اللي المفروض تشوفيه: (1) كل مضاعفة للعدد بتضيف نفس المقدار تقريبًا، يعني الفايدة بتقلّ مع الزيادة؛ (2) الصور القوية بتوصل لنفس المنفعة بعدد أقل.

In [ ]:
import numpy as np

b, lam = 0.90, 0.010                      # invented numbers, for illustration only
sizes = np.array([0, 125, 250, 500, 1000, 2000, 3168])
print(f"{'images':>7} | {'w=0.5':>7} | {'w=1.0':>7} | {'w=1.5':>7}")
for n in sizes:
    print(f"{n:>7} | " + " | ".join(f"{b + lam * np.log1p(w * n):7.4f}" for w in (0.5, 1.0, 1.5)))

<a id="s8"></a>

---
## 8. Bootstrap / Lower Bound

**الفكرة:** ranker واحد بيدّي رقم واحد لكل صورة، ومش بيقول واثق فيه قد إيه. عشان نعرف الثقة بنعمل **bootstrap**:

1. من الـ 120 مجموعة train، بنسحب 120 مجموعة **مع الإرجاع** (فبعض المجموعات بتتكرر وبعضها بيغيب).
2. بندرّب ranker كامل على العيّنة دي، بنفس الإعدادات ونفس early stopping.
3. بنكرر **200 مرة**.

دلوقتي لكل صورة 200 تقدير لفايدتها. **الحد الأدنى 95%** هو القيمة اللي 5% بس من النسخ تحتها (5th percentile). لو الحد ده فوق الصفر، يبقى تقريبًا كل النسخ متفقة إن الصورة مفيدة.

**ليه الحد الأدنى مش المتوسط:** المتوسط ممكن يطلع موجب بالصدفة. الحد الأدنى بيطلب دليل، فالصورة المشكوك فيها ما بتتاخدش.

الـ ranker والـ 200 نسخة بيتحفظوا في ملف واحد، فالاختيار بيقرا الـ fit بدل ما يعيده، والملف بيرفض لو محتواه أو مصدره اتغير.

| بيعمل إيه | فين في الكود |
|---|---|
| السحب مع الإرجاع وتدريب النسخ (جوه `fit_ranker`) | `fit_ranker` — [`scripts/asism_v2/pipeline.py:135`](../scripts/asism_v2/pipeline.py) |
| تدريب نسخة واحدة | `_train` — [`scripts/asism_v2/pipeline.py:276`](../scripts/asism_v2/pipeline.py) |
| حساب الحد الأدنى لكل صورة (`own_lower`) | `progressive_select` — [`scripts/asism_v2/stopping.py:52`](../scripts/asism_v2/stopping.py) |
| حفظ الـ ranker والنسخ | `save_fitted` — [`scripts/asism_v2/persist.py:34`](../scripts/asism_v2/persist.py) |
| تحميله مع فحص المصدر | `load_fitted` — [`scripts/asism_v2/persist.py:48`](../scripts/asism_v2/persist.py) |

**الإعدادات المعتمدة:** 200 نسخة، Adam، learning rate 0.03، من غير weight decay، حد أقصى 5,000 epoch و patience 200، والـ fit على درجات مطبّعة.

**الداخل:** قياسات train و validation.
**الخارج:** `ranker_fit_seed42.json`.

**من الكود الفعلي** — [`scripts/asism_v2/pipeline.py`](../scripts/asism_v2/pipeline.py)، السطور 251–257:

```python
for b in range(bootstrap):
    rows = torch.from_numpy(np.random.default_rng([seed, b]).integers(0, n_train, n_train))
    resampled = tuple(t[rows] for t in pooled["train"])
    member, _, _, limit = _train(resampled, targets["train"][rows], pooled["validation"],
                                 targets["validation"], seed + 1 + b, *config)
    members_at_limit += int(limit)
    ensemble.append(member)
```

<a id="s9"></a>

---
## 9. Adaptive Stopping

**الفكرة:** دي القاعدة اللي بتحدد **كام صورة**. مفيش K ولا نسبة ولا حد أدنى للفايدة:

1. جوه كل class، الصور بتترتب بالحد الأدنى بتاعها (أحسن صورة الأول).
2. في كل خطوة، كل class لسه شغّال **بيعرض** صورته الجاية.
3. لكل عرض بنحسب: لو ضفنا الصورة دي للمجموعة الحالية، `U` هتزيد قد إيه؟ بنحسبها في الـ 200 نسخة وناخد الحد الأدنى 95%.
4. الـ class اللي حد عرضه **≤ 0** بيقف نهائيًا.
5. من الباقيين، بناخد صاحب أعلى حد أدنى، ونكرر.

كل class ممكن يقف عند عدد مختلف، وكل خطوة وكل وقفة بتتسجل بسببها (`trajectory` و `stops`).

**ثلاث نتايج ممكنة، وكلهم صالحين:** `subset` (جزء من الصور)، `all` (كل الصور)، `none` (ولا صورة).

**الثبات (stability):** الـ fit والاختيار كله بيتعاد بخمس fit seeds (42–46)، وبيتكتب: أقل وأكبر عدد، وتطابق كل اختيار مع الاختيار المعتمد (Jaccard). ده **تقرير بس**: الاختيار المستخدم هو بتاع seed 42، ومحدش بيختار من الخمسة.

**القاعدة بترفض لو النموذج ما يقدرش يحدد أثر الحجم** (كل المجموعات بحجم واحد) أو أثر الـ class، ولو النسخ أقل من الحد الأدنى للـ bootstrap.

| بيعمل إيه | فين في الكود |
|---|---|
| قاعدة الوقوف كلها | `progressive_select` — [`scripts/asism_v2/stopping.py:52`](../scripts/asism_v2/stopping.py) |
| الزيادة في `U` | `_utility` — [`scripts/asism_v2/stopping.py:48`](../scripts/asism_v2/stopping.py) |
| الثبات على الـ fit seeds | `selection_stability` — [`scripts/asism_v2/gates.py:160`](../scripts/asism_v2/gates.py) |

**الداخل:** الـ ranker المحفوظ والـ pool الآمن.
**الخارج:** قايمة الصور المختارة، العدد لكل class، وسجل الخطوات.

**من الكود الفعلي** — [`scripts/asism_v2/stopping.py`](../scripts/asism_v2/stopping.py)، السطور 99–112:

```python
def offer(name):
    i = order[name][position[name]]
    gains = [_utility(lams[m], totals[m] + weights[m, i]) - _utility(lams[m], totals[m])
             for m in range(len(models))]
    return i, gains[0], float(np.quantile(gains[1:], LOWER_BOUND_QUANTILE))

while active:
    offers = {name: offer(name) for name in sorted(active)}
    for name, (i, point, lower) in offers.items():
        if lower <= 0:
            stops[name] = {"reason": "lower 95% bound of marginal utility <= 0",
                           "at_count": position[name], "next_image_id": ids[i],
                           "marginal_point": point, "marginal_lower_bound": lower}
            active.discard(name)
```

### مثال توضيحي 2: الحد الأدنى 95% ونقطة الوقوف

**الأرقام مخترعة برضه.** class واحد فيه 300 صورة؛ وزنها الحقيقي نازل من +1 لـ −0.5. بنعمل 200 «نسخة ranker» كل واحدة شايفة الوزن بضوضاء، وبعدها نطبّق نفس فكرة القاعدة: نرتّب بالحد الأدنى، ونقف عند أول صورة حدها الأدنى مش فوق الصفر.

اللي المفروض تشوفيه: القاعدة بتاخد **أقل** من عدد الصور اللي متوسطها موجب، لأنها عايزة تقريبًا كل النسخ تتفق. كل ما الضوضاء تزيد، العدد يقلّ. ده مبسّط: الكود الحقيقي بيحسب الزيادة في `U` نفسها لكل نسخة ولكل الـ classes مع بعض.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
true_w = np.linspace(1.0, -0.5, 300)                      # invented: 200 useful images, 100 harmful
for noise in (0.05, 0.20, 0.50):
    estimates = true_w + rng.normal(0, noise, size=(200, 300))    # 200 bootstrap rankers
    lower = np.quantile(estimates, 0.05, axis=0)                   # each image's lower 95% bound
    order = np.argsort(-lower)                                     # the class offers its best first
    stop = int(np.argmax(lower[order] <= 0)) if (lower <= 0).any() else len(order)
    print(f"noise {noise:.2f}: mean weight > 0 for {(estimates.mean(0) > 0).sum():3d} images, "
          f"the rule keeps {stop:3d}; of those truly useful: {(true_w[order[:stop]] > 0).sum():3d}")

<a id="s10"></a>

---
## 10. C / D

**الفكرة:** عشان نعرف إن الفايدة جاية من **أنهي صور** مش من **العدد**، لازم نقارن C بحالة فيها نفس العدد بس عشوائي.

- **C** = اللي قاعدة الوقوف اختارته.
- **D** = لكل class، نفس عدد C، مسحوب عشوائي من **نفس الـ pool الآمن**.

فالفرق الوحيد بين C و D هو الترتيب المتعلَّم.

**سحبة D لكل seed:** لكل seed من الـ 20 بتوع Stage 4 فيه سحبة D مستقلة، عشان النتيجة ما تبقاش مربوطة بسحبة عشوائية واحدة محظوظة أو منحوسة.

**لو النتيجة `all` أو `none`:** مفيش D، لأن C بقت هي B (أو A) بالتعريف. بيتدرب A و B بس، والنتيجة نفسها هي اللي بتتكتب.

| `selection_outcome` | البروتوكول | اللي بيتدرب | المقارنة التأكيدية |
|---|---|---|---|
| `subset` | `asism_v2_learned` | A, B, C, D | C ضد D |
| `all` / `none` | `asism_v2_learned_all_or_none` | A, B | مفيش |

| بيعمل إيه | فين في الكود |
|---|---|
| تحديد النتيجة وبناء C و D | `build_selection` — [`scripts/asism_v2/selection_files.py:50`](../scripts/asism_v2/selection_files.py) |
| سحبة D بنفس عدد C لكل class | `draw_matched_random` — [`scripts/asism_v2/selection_files.py:40`](../scripts/asism_v2/selection_files.py) |
| كتابة الملفات والـ manifest | `write_selection` — [`scripts/asism_v2/selection_files.py:71`](../scripts/asism_v2/selection_files.py) |
| تعريف البروتوكولين والمقارنات | `ConditionProtocol` — [`scripts/utils/ham10000_conditions.py:49`](../scripts/utils/ham10000_conditions.py) |

**الداخل:** الـ ranker المحفوظ، الـ pool، وقايمة seeds الـ Stage 4.
**الخارج:** `c_selected.csv`، `d_selected_seed42.csv` … `d_selected_seed61.csv`، و `asism_v2_learned_selection_manifest.json`.

**من الكود الفعلي** — [`scripts/asism_v2/selection_files.py`](../scripts/asism_v2/selection_files.py)، السطور 40–47:

```python
def draw_matched_random(pool: pd.DataFrame, counts: dict[str, int], seed: int,
                        base: int = D_DRAW_SEED_BASE) -> list[str]:
    rng = np.random.default_rng([int(base), int(seed)])
    drawn = []
    for dx in sorted(counts):
        ids = np.array(sorted(pool.loc[pool["dx"].astype(str) == dx, "image_id"].astype(str)))
        drawn += list(rng.choice(ids, size=int(counts[dx]), replace=False))
    return drawn
```

<a id="s11"></a>

---
## 11. E4 — تجربة مستقلة لفحص الكمية

> **E4 تجربة مستقلة.** مش جزء من تدريب الـ ranker، ومش بتدخل في قاعدة الوقوف، والعدد اللي ASISM بيختاره **مش** جاي منها. دورها في v2: فحص خارجي نحط جنبه العدد اللي ASISM طلّعه.

**السؤال:** لما نزوّد الصور الصناعية، الأداء بيتغير إزاي؟ وهل التغيير ده ظاهر فوق الضوضاء؟

**التصميم (متوافق عليه يوم 2026-10-02 قبل الكود):**

| البند | القيمة |
|---|---|
| الـ recipe | بتاع Stage 4 نفسه: 512 px، 3,000 خطوة |
| الأحجام | 0، 250، 500، 1,000، 2,000، وكل الـ pool |
| السحب | سلسلتين متداخلتين: كل سلسلة ترتيب عشوائي للـ pool، والحجم `s` هو أول `s` صورة |
| الـ seeds | الحقيقي بس: 42–51. باقي الأحجام: سلسلتين × 42–46 |
| العدد | 60 تدريب، 10 لكل حجم |
| المقياس | macro AUROC على `asism_tuning_heldout` |
| القاعدة | GO / COARSE / NO من اختبار Welch، مكتوبة قبل التشغيل |

**الفرق عن قياس الـ ranker:**

| | قياس الـ ranker (القسم 7) | E4 |
|---|---|---|
| الهدف | يعلّم الـ ranker | يرسم منحنى الكمية |
| الـ recipe | صغير: 224 px، 300 خطوة | الكامل: 512 px، 3,000 خطوة |
| الصور | مجموعات مصممة (أحجام ونسب وميل) | عشوائي متداخل |
| العدد | 1,000 تدريب | 60 تدريب |

**الحالة:** الـ 60 تدريب خلصوا على الـ pod. ملف التشغيلات مش موجود على اللابتوب وقت بناء النوتبوك، **فنتيجة E4 مش مكتوبة هنا**.

| بيعمل إيه | فين في الكود |
|---|---|
| السلسلتين المتداخلتين | `build_chains` — [`scripts/followup/ham10000_e4_quantity_curve.py:91`](../scripts/followup/ham10000_e4_quantity_curve.py) |
| الـ 60 خلية | `plan_cells` — [`scripts/followup/ham10000_e4_quantity_curve.py:105`](../scripts/followup/ham10000_e4_quantity_curve.py) |
| التأكد إن الـ recipe هو بتاع Stage 4 بالظبط | `check_stage4_recipe` — [`scripts/followup/ham10000_e4_quantity_curve.py:128`](../scripts/followup/ham10000_e4_quantity_curve.py) |
| قاعدة القرار | `decide` — [`scripts/followup/ham10000_e4_quantity_curve.py:209`](../scripts/followup/ham10000_e4_quantity_curve.py) |
| العدد `q*` اللي بيتحط جنب عدد ASISM | `q_star` — [`scripts/followup/ham10000_e4_consequences.py:36`](../scripts/followup/ham10000_e4_consequences.py) |
| التصميم | [`docs/ham10000_e4_quantity_design.md`](../docs/ham10000_e4_quantity_design.md) |

<a id="s12"></a>

---
## 12. Stage 4

**الفكرة:** ده الاختبار الحقيقي. بندرّب الـ classifier الكامل على أربع حالات، **والفرق الوحيد بينهم هو داتا التدريب**:

| الحالة | بيتدرب على |
|---|---|
| A | `classifier_train` بس (1,641) |
| B | الحقيقي + كل الـ 3,168 |
| C | الحقيقي + اختيار ASISM |
| D | الحقيقي + عشوائي بنفس عدد C لكل class |

**الـ recipe (مجمّد):** DenseNet-121 (ImageNet)، 512 px، 3,000 خطوة، batch 32، learning rate 1e-4، softmax + cross-entropy، من غير class weighting، **20 seed** (42–61). عدد الخطوات ثابت في كل الحالات، فمفيش حالة بتاخد تدريب أكتر عشان داتاها أكبر.

**فحص العدالة قبل أي جدول:** الـ aggregator بيقارن إعدادات كل التدريبات. لو أي حاجة غير الداتا اختلفت بين حالتين، بيرفض يكتب الجدول. وكمان بيتأكد إن C و D بنفس العدد، وإن نتيجة الاختيار في الـ manifest مطابقة للبروتوكول.

**ملحوظتين:** التدريب مش bit-deterministic، فالنتايج بتتقارن كتوزيعات على الـ seeds. والـ classifier بيتدرب من غير augmentation؛ ده حد مسجّل من حدود البروتوكول.

| بيعمل إيه | فين في الكود |
|---|---|
| تجميع داتا الحالة (حقيقي + ملف الصناعي بتاعها) | `build_condition_records` — [`scripts/classify/ham10000_train_conditions.py:117`](../scripts/classify/ham10000_train_conditions.py) |
| اختيار ملف الصناعي حسب الحالة والـ seed (D ليها ملف لكل seed) | `resolve_synthetic_manifest` — [`scripts/classify/ham10000_train_conditions.py:107`](../scripts/classify/ham10000_train_conditions.py) |
| تدريب حالة × seed | `run` — [`scripts/classify/ham10000_train_conditions.py:158`](../scripts/classify/ham10000_train_conditions.py) |
| حلقة التدريب | `train_classifier` — [`scripts/utils/ham10000_classifier.py:121`](../scripts/utils/ham10000_classifier.py) |
| فحص إن الداتا بس هي اللي اختلفت | `check_fairness` — [`scripts/classify/ham10000_aggregate_conditions.py:139`](../scripts/classify/ham10000_aggregate_conditions.py) |
| فحص إن نتيجة الاختيار مطابقة للبروتوكول | `check_selection_outcome` — [`scripts/classify/ham10000_aggregate_conditions.py:227`](../scripts/classify/ham10000_aggregate_conditions.py) |
| الإعدادات والمسارات | [`configs/ham10000_asism_v2_learned_stage4.yaml`](../configs/ham10000_asism_v2_learned_stage4.yaml) |

**الداخل:** `classifier_train`، `all_candidates.csv`، `c_selected.csv`، `d_selected_seed*.csv`.
**الخارج:** لكل حالة × seed في `outputs/ham10000/stage4_asism_v2_learned/ham-stratified-v1/<حالة>/seed<رقم>/`: النموذج، التوقعات على `classifier_val`، و `run_manifest.json`.

**من الكود الفعلي** — [`scripts/classify/ham10000_train_conditions.py`](../scripts/classify/ham10000_train_conditions.py)، السطور 107–114:

```python
def resolve_synthetic_manifest(config, condition: str, seed: int | None):
    """The condition's synthetic manifest path, with {seed} filled for a per-seed control (D)."""
    manifest = config.conditions[condition].synthetic_manifest
    if manifest and "{seed}" in str(manifest):
        if seed is None:
            raise Stage4ConfigError(f"condition {condition!r} has a per-seed manifest; a seed is required")
        manifest = str(manifest).replace("{seed}", str(int(seed)))
    return manifest
```

<a id="s13"></a>

---
## 13. Metrics

**الفكرة:** الداتا مش متوازنة (`nv` هو الأغلبية)، فالـ accuracy العادية بتخدع: نموذج بيقول `nv` دايمًا ياخد رقم عالي. عشان كده المقياس الأساسي هو **balanced accuracy**.

| المقياس | بيقيس إيه | الأحسن |
|---|---|---|
| **Balanced accuracy** (الأساسي والتأكيدي) | متوسط الـ recall على الأنواع السبعة؛ كل نوع ليه نفس الوزن | الأعلى |
| Macro-F1 | متوسط F1 على الأنواع | الأعلى |
| Macro AUROC | قدرة النموذج يرتّب الصح فوق الغلط، من غير عتبة | الأعلى |
| Accuracy | نسبة الصح الكلية | الأعلى |
| **Macro average precision** (جديد في v2) | زي AUROC بس أدق مع الأنواع النادرة | الأعلى |
| **Brier score** (جديد في v2) | بُعد الاحتمالات عن الحقيقة. 0 مثالي، 2 واثق وغلطان | **الأقل** |
| **ECE top-label** (جديد في v2) | هل ثقة النموذج صادقة؟ لو قال 80% يبقى صح 80% من المرات. 15 خانة متساوية العرض | **الأقل** |

**الثلاثة الجداد لـ v2 بس.** تقارير v1 بتطلّع نفس الأربعة اللي طلّعتهم بالظبط، وفيه اختبار بيتأكد من ده. عدد الخانات في ECE ثابت (15) لأن تغييره بيغيّر الرقم.

| بيعمل إيه | فين في الكود |
|---|---|
| Balanced accuracy | `balanced_accuracy` — [`scripts/utils/ham10000_metrics.py:78`](../scripts/utils/ham10000_metrics.py) |
| Macro average precision | `macro_average_precision` — [`scripts/utils/ham10000_metrics.py:127`](../scripts/utils/ham10000_metrics.py) |
| Brier | `multiclass_brier_score` — [`scripts/utils/ham10000_metrics.py:134`](../scripts/utils/ham10000_metrics.py) |
| ECE | `top_label_ece` — [`scripts/utils/ham10000_metrics.py:146`](../scripts/utils/ham10000_metrics.py) |
| أنهي مقاييس لأنهي بروتوكول | `metric_functions` — [`scripts/eval/ham10000_compare_conditions.py:79`](../scripts/eval/ham10000_compare_conditions.py) |

**من الكود الفعلي** — [`scripts/utils/ham10000_metrics.py`](../scripts/utils/ham10000_metrics.py)، السطور 154–162:

```python
confidence = probabilities.max(axis=1)
correct = (probabilities.argmax(axis=1) == y_true).astype(np.float64)
edges = np.linspace(0.0, 1.0, int(n_bins) + 1)
error = 0.0
for lower, upper in zip(edges[:-1], edges[1:]):
    in_bin = (confidence > lower) & (confidence <= upper) if lower > 0 else (confidence >= lower) & (confidence <= upper)
    if in_bin.any():
        error += in_bin.mean() * abs(correct[in_bin].mean() - confidence[in_bin].mean())
return float(error)
```

<a id="s14"></a>

---
## 14. Final Comparison

**الفكرة:** فرق بين رقمين مش نتيجة لوحده؛ لازم نعرف هل هو أكبر من الصدفة.

**1) Bootstrap على مستوى الـ lesion.** بنسحب lesions مع الإرجاع 2,000 مرة ونعيد حساب المقياس كل مرة، فبيطلع فاصل ثقة 95%. السحب بالـ lesion مش بالصورة، لأن صور نفس الـ lesion مش مستقلة عن بعض. وللفرق بين حالتين السحب «paired»: نفس الـ lesions للحالتين.

**2) عيلتين من المقارنات.**

| العيلة | فيها إيه | التصحيح |
|---|---|---|
| **تأكيدية** | مقارنة واحدة: **C ضد D على balanced accuracy** | Holm |
| استكشافية | كل الباقي: C ضد B، C ضد A، B ضد A، D ضد A، D ضد B، كل المقاييس التانية، والـ recall لكل class | Benjamini–Hochberg |

المقارنة التأكيدية مكتوبة قبل ما أي نتيجة تتشاف. إضافة المقاييس الجديدة كبّرت العيلة الاستكشافية بس، وما لمستش التأكيدية.

**3) C ضد D بتختبر إيه وما بتختبرش إيه.** بتختبر **أنهي صور** بس، لأن العدد متساوي. **كام صورة** بتتقري من C ضد A و C ضد B، ومن عدد C جنب منحنى E4. دي قراءات استكشافية، والإحصائية التأكيدية للكمية لسه قرار عند المشرف.

**4) مصدرين للتوقعات.**

- `--source classifier_val`: مكتوب عليه **MONITORING** في كل مكان. مش التقييم النهائي ومش بيقرر حاجة.
- `--source final`: على `final_eval_heldout`. **مش متاح دلوقتي**: القسم ده اتقرا مرة في v1، وسياسة استخدامه في v2 مستنية قرار المشرف.

**5) عدم اليقين من الـ seeds.** الفاصل اللي فوق بيسحب lesions بس، فتغيّر النتيجة من seed لـ seed ما بيدخلش فيه، وده تغيّر كبير في الـ recipe ده. عشان كده فيه تحليلين معلنين مسبقًا بيتكتبوا **جنب** الفاصل الأساسي مش بداله: اختبار Welch على balanced accuracy لكل seed (الوحدة هي التدريبة)، و bootstrap على مستويين بيسحب lesions و seeds مع بعض.

| بيعمل إيه | فين في الكود |
|---|---|
| المقارنة كلها (فواصل، فروق، العيلتين) | `compare` — [`scripts/eval/ham10000_compare_conditions.py:176`](../scripts/eval/ham10000_compare_conditions.py) |
| فاصل الثقة بالـ lesion | `patient_level_bootstrap` — [`scripts/utils/metrics.py:253`](../scripts/utils/metrics.py) |
| فرق paired بين حالتين | `paired_bootstrap_difference` — [`scripts/utils/metrics.py:302`](../scripts/utils/metrics.py) |
| Holm | `holm_bonferroni` — [`scripts/utils/metrics.py:360`](../scripts/utils/metrics.py) |
| Benjamini–Hochberg | `benjamini_hochberg` — [`scripts/utils/metrics.py:404`](../scripts/utils/metrics.py) |
| تجميع توقعات `classifier_val` وتشغيل المقارنة | `run` — [`scripts/followup/ham10000_asism_v2_compare.py:70`](../scripts/followup/ham10000_asism_v2_compare.py) |
| تحليل الثبات على الـ seeds | `robustness` — [`scripts/eval/ham10000_seed_robustness.py:87`](../scripts/eval/ham10000_seed_robustness.py) |

**الداخل:** ملفات التوقعات لكل حالة × seed.
**الخارج:** تقرير JSON وجدول CSV فيهم كل مقياس بفاصله، وكل فرق بـ p المصحّحة وحجم الأثر.

**من الكود الفعلي** — [`scripts/eval/ham10000_compare_conditions.py`](../scripts/eval/ham10000_compare_conditions.py)، السطور 205–210:

```python
confirmatory, exploratory = {}, {}
for left, right in protocol.confirmatory:
    confirmatory[f"{left}_vs_{right}:{PRIMARY_METRIC}"] = _difference(left, right, metrics[PRIMARY_METRIC])
for left, right in protocol.exploratory:
    for name, metric_fn in metrics.items():
        exploratory[f"{left}_vs_{right}:{name}"] = _difference(left, right, metric_fn)
```

---
## القواعد اللي المشروع ماشي بيها

- أي حد (threshold) بيتكتب قبل التشغيل اللي هيتحكم عليه بيه، وما بيترخّاش بعده.
- أي تغيير هو تعديل مكتوب بتاريخه **قبل** التشغيل اللي بعده.
- النتايج القديمة ما بتتعدّلش وما بتتمسحش؛ الجديد بيروح فولدرات جديدة.
- `final_eval_heldout` ما بيتقراش من أي كود جديد، و `classifier_val` ما بيتستخدمش في اختيار الصور.
- النتيجة السلبية بتتكتب زي ما هي.

## إيه اللي ممكن يوقّف v2، وكله نتيجة مش خطأ

| فين | الشرط | لو ما اتحققش |
|---|---|---|
| G1 | ثبات متوسطات المجموعات ≥ 0.80 | مفيش تعلّم من القياس |
| accept | Spearman ≥ 0.50، p ≤ 0.05، وأحسن من «الحجم والـ class بس» | الترتيب المتعلَّم ما بيتستخدمش |
| select | النتيجة `all` أو `none` | مفيش C و D؛ بيتدرب A و B بس |

**قراءتي أنا، مش نتيجة:** التشخيص القديم لقى إن القياس بالـ recipe الصغير ضوضاؤه عالية. v2 بيختلف في إن الأحجام متنوعة وكل مجموعة بتتكرر 5 مرات، بس ده ما يضمنش إن G1 يعدّي. فشل G1 أو accept احتمال حقيقي.

## اللي لسه مفتوح

| القرار | عند مين |
|---|---|
| الموافقة على القياس (GPU) | ولاء |
| الموافقة على Stage 4 | ولاء، بعد قراءة الاختيار |
| سياسة الـ test set في v2 | المشرف |
| الإحصائية التأكيدية للكمية | المشرف |
| نطاق الرسالة | المشرف |